# Extending Models

This tutorial shows how to create a custom model by extending [`BaseModel`](https://instadeepai.github.io/alf/api/alf_core/model/). We'll implement a simple polynomial regression model as an example.

## 1. Imports

In [ ]:
import numpy as np
from alf_core.dataclasses import Candidate, LabelledCandidates, Modality, Predictions
from alf_core.model.base_model import BaseModel

## 2. Define Custom Model

Implement the required abstract methods: `featurise()`, `train()`, `predict()`, and `sample()`. The `featurise()` method converts [`Candidate`](https://instadeepai.github.io/alf/api/alf_core/dataclasses/candidate/) objects to feature vectors, `train()` fits the model on [`LabelledCandidates`](https://instadeepai.github.io/alf/api/alf_core/dataclasses/labelled_candidates/), and `predict()` returns [`Predictions`](https://instadeepai.github.io/alf/api/alf_core/dataclasses/predictions/) with means and variances.

In [ ]:
class PolynomialModel(BaseModel):
    """Simple polynomial regression model for demonstration."""

    def __init__(self, degree: int = 2):
        self.degree = degree
        self.coefficients = None

    def featurise(self, inputs: list[Candidate]) -> np.ndarray:
        """Convert candidates to feature vectors."""
        # Extract numeric data from candidates
        X = np.array([c.data for c in inputs])
        # Create polynomial features
        features = np.column_stack([X**i for i in range(1, self.degree + 1)])
        return features

    def train(self, train_data: LabelledCandidates, val_data: LabelledCandidates) -> None:
        """Train polynomial regression using least squares."""
        X = self.featurise(train_data.candidates)
        y = train_data.labels
        # Fit using least squares
        self.coefficients = np.linalg.lstsq(X, y, rcond=None)[0]

    def predict(self, candidate_points: list[Candidate]) -> Predictions:
        """Make predictions with the trained model."""
        X = self.featurise(candidate_points)
        means = X @ self.coefficients
        # Simple variance estimate from residuals (optional)
        variances = np.ones_like(means) * 0.1  # Placeholder
        return Predictions(means=means, variances=variances)

    def sample(self, condition=None) -> list[Candidate]:
        """Sample random candidates (for generative models)."""
        # For this simple model, sample random x values
        x_samples = np.random.uniform(-10, 10, size=10)
        return [Candidate(data=x, modality=Modality.TABULAR) for x in x_samples]

The four methods map directly onto the `BaseModel` contract:

- **`featurise`** turns each `Candidate`'s raw value into the polynomial feature vector `[x, x², …, x^degree]` the model regresses on.
- **`train`** solves the least-squares fit for the polynomial coefficients.
- **`predict`** returns a `Predictions` object — the mean from the fitted polynomial, plus a placeholder variance (a real model would estimate this).
- **`sample`** is only needed when the model acts as a generator; here it returns random candidates.

See the [Model Roles tutorial](model_roles.ipynb) for how these methods are used when the same model plays the Oracle, Surrogate, or Generator role.

## 3. Usage Example

In [ ]:
# Create training data
X_train = [Candidate(data=x, modality=Modality.TABULAR) for x in [1.0, 2.0, 3.0, 4.0]]
y_train = np.array([2.0, 5.0, 10.0, 17.0])  # y = x^2 + 1
train_data = LabelledCandidates(candidates=X_train, labels=y_train)

# Create validation data
X_val = [Candidate(data=x, modality=Modality.TABULAR) for x in [2.5, 3.5]]
y_val = np.array([7.25, 13.25])
val_data = LabelledCandidates(candidates=X_val, labels=y_val)

# Train the model
model = PolynomialModel(degree=2)
model.train(train_data, val_data)

# Make predictions
test_points = [Candidate(data=5.0, modality=Modality.TABULAR)]
predictions = model.predict(test_points)
print(f"Prediction for x=5: {predictions.means[0]:.2f}")

# Sample new candidates
samples = model.sample()
print(f"Sampled {len(samples)} candidates")

## Key Points

- **Required methods**: `featurise()`, `train()`, `predict()`, `sample()` must all be implemented
- **featurise()**: Convert candidates to features your model can process
- **train()**: Fit the model using training and validation data
- **predict()**: Return `Predictions` object with means and optionally variances
- **sample()**: For generative models; return list of new `Candidate` objects
- **Optional**: Override `get_training_summary_metrics()` to return training metrics
- **Optional**: Override `cleanup()` to clean up temporary resources

See [Model Roles Tutorial](model_roles.ipynb) to learn how to use your model as an Oracle, Surrogate, or Generator.